# 🔍 DocForge-VLM: Document Forgery Detection with Fine-tuned Vision-Language Models

Fine-tuning **Qwen2-VL-2B-Instruct** with **QLoRA** (4-bit NF4) for identity document tampering detection on the **SIDTD** benchmark.

**Author**: [Your Name]
**Repository**: [GitHub](https://github.com/CRGaming78/DocForge-VLM)

| Component | Details |
|---|---|
| Base Model | Qwen2-VL-2B-Instruct |
| Method | QLoRA (4-bit NF4 + LoRA r=16) |
| Dataset | SIDTD (Synthetic Identity Document Tampering Detection) |
| Platform | Kaggle T4 GPU |

In [ ]:
%%capture
!pip install -q transformers>=4.45.0 peft>=0.13.0 bitsandbytes>=0.43.0 trl>=0.12.0 accelerate>=0.34.0 qwen-vl-utils scikit-learn seaborn huggingface_hub
!git clone https://github.com/Oriolrt/SIDTD_Dataset.git /tmp/SIDTD_Dataset
!cd /tmp/SIDTD_Dataset && pip install -e .

In [ ]:
import os, gc, json, random, glob
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from transformers import (
    Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTConfig, SFTTrainer

# HuggingFace login
import os
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except: pass
if hf_token:
    login(token=hf_token)
    print("✅ Logged in to HuggingFace")

# Seeds
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
OUTPUT_DIR = "./docforge_vlm_output"
RESULTS_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# QLoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training
BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4
EPOCHS = 3
WARMUP_RATIO = 0.1
MAX_SEQ_LENGTH = 1024

## 📦 Dataset: SIDTD
Downloading the Synthetic Identity Document Tampering Detection dataset using the official Python package.

In [ ]:
# Download SIDTD dataset using official package
try:
    from SIDTD.data.DataLoader.Datasets import SIDTD as SIDTDDataset
    data = SIDTDDataset(download_original=False, custom_path_to_download="./data").download_dataset("templates")
    print("✅ SIDTD dataset downloaded successfully")
except Exception as e:
    print(f"⚠️ SIDTD package download failed: {e}")
    print("Trying alternative download...")
    # Fallback: Try to use the dataset from Kaggle input
    pass

# Find all images
DATA_ROOT = Path("./data")

def find_images(directory):
    """Recursively find all image files."""
    images = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        images.extend(Path(directory).rglob(ext))
    return sorted(set(images))

# Search for the dataset structure
all_images = find_images(DATA_ROOT)
print(f"Total images found: {len(all_images)}")

# Classify images based on directory structure
images, labels = [], []
for img_path in all_images:
    path_str = str(img_path).lower()
    if any(kw in path_str for kw in ["forg", "fraud", "fake", "tamper", "alter"]):
        labels.append(1)  # Forged/Tampered
    elif any(kw in path_str for kw in ["bona", "real", "genuine", "authentic", "original"]):
        labels.append(0)  # Authentic/Bonafide
    else:
        continue  # Skip ambiguous images
    images.append(img_path)

print(f"\nClassified images: {len(images)}")
print(f"  Authentic (bonafide): {labels.count(0)}")
print(f"  Forged (tampered):    {labels.count(1)}")

# Show sample images
if images:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Sample Documents from SIDTD", fontsize=16, fontweight="bold")
    auth_imgs = [img for img, lbl in zip(images, labels) if lbl == 0]
    forg_imgs = [img for img, lbl in zip(images, labels) if lbl == 1]
    for i in range(4):
        if i < len(auth_imgs):
            axes[0, i].imshow(Image.open(auth_imgs[i]))
            axes[0, i].set_title("AUTHENTIC", color="green", fontweight="bold")
        axes[0, i].axis("off")
        if i < len(forg_imgs):
            axes[1, i].imshow(Image.open(forg_imgs[i]))
            axes[1, i].set_title("FORGED", color="red", fontweight="bold")
        axes[1, i].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "sample_documents.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 📊 Train/Val/Test Split
Stratified 70/15/15 split to ensure balanced class distribution.

In [ ]:
# Stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, stratify=labels, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)

print(f"Train: {len(X_train)} ({sum(y_train)} forged, {len(y_train)-sum(y_train)} authentic)")
print(f"Val:   {len(X_val)} ({sum(y_val)} forged, {len(y_val)-sum(y_val)} authentic)")
print(f"Test:  {len(X_test)} ({sum(y_test)} forged, {len(y_test)-sum(y_test)} authentic)")

# Prompt templates
SYSTEM_PROMPT = (
    "You are an expert in document forensics specializing in identity document "
    "verification. Analyze the provided document image for signs of forgery, "
    "tampering, or digital manipulation. Provide your verdict as AUTHENTIC or "
    "TAMPERED, followed by detailed reasoning."
)

USER_PROMPTS = [
    "Analyze this identity document for signs of forgery or tampering. Is it authentic or tampered?",
    "Examine this document image carefully. Determine if it is genuine or has been digitally manipulated.",
    "You are reviewing this identity document. Check for any signs of tampering or digital manipulation.",
    "Inspect this ID card for authenticity. Is it a real document or a forgery?",
    "Conduct a forensic analysis of this document image. Can you identify any tampering?",
    "Assess this document for signs of digital alteration. State your verdict.",
    "Review the provided identity document and determine its authenticity.",
    "Look closely at the details of this document. Does it appear genuine or altered?",
]

AUTHENTIC_RESPONSES = [
    "VERDICT: AUTHENTIC\n\nAnalysis: After careful examination, this document appears to be genuine. The fonts are consistent throughout all text fields, the alignment of elements is proper, and there are no visible signs of digital tampering or splicing artifacts.",
    "VERDICT: AUTHENTIC\n\nAnalysis: This identity document passes forensic inspection. No inconsistencies detected in font styles, character spacing, or image compression patterns. The photograph region shows natural integration with the document background.",
    "VERDICT: AUTHENTIC\n\nAnalysis: The document exhibits consistent physical characteristics. Text fields show uniform typography, edge transitions are natural, and no splicing or copy-move artifacts are detected.",
    "VERDICT: AUTHENTIC\n\nAnalysis: No evidence of manipulation found. The document's layout, text alignment, and color consistency are within expected parameters for a genuine identity document.",
    "VERDICT: AUTHENTIC\n\nAnalysis: This document appears unaltered. The text fields maintain consistent font weight, kerning, and baseline alignment. Background patterns show no interruption.",
]

TAMPERED_RESPONSES = [
    "VERDICT: TAMPERED\n\nAnalysis: This document shows signs of digital manipulation. There are inconsistencies in the font style and weight within text fields. The character spacing appears irregular, suggesting text replacement.",
    "VERDICT: TAMPERED\n\nAnalysis: Evidence of forgery detected. The text region shows subtle compression artifacts inconsistent with the surrounding area, indicating digital modification of text fields.",
    "VERDICT: TAMPERED\n\nAnalysis: This identity document has been altered. There is visible misalignment in certain text fields, and the font rendering differs from the expected template.",
    "VERDICT: TAMPERED\n\nAnalysis: Forensic analysis reveals manipulation. The document exhibits inconsistent text baselines, slight color differences in modified fields, and unnatural edge transitions.",
    "VERDICT: TAMPERED\n\nAnalysis: The document is not genuine. Anomalies include mismatched character rendering in personal data fields and localized compression artifacts consistent with text replacement.",
]

def format_sample(image_path, label):
    """Format a single sample for Qwen2-VL training."""
    user_prompt = random.choice(USER_PROMPTS)
    response = random.choice(AUTHENTIC_RESPONSES) if label == 0 else random.choice(TAMPERED_RESPONSES)
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": str(image_path)},
            {"type": "text", "text": user_prompt},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": response}]},
    ]
    return {"messages": messages, "label": label}

# Create datasets
train_data = [format_sample(img, lbl) for img, lbl in zip(X_train, y_train)]
val_data = [format_sample(img, lbl) for img, lbl in zip(X_val, y_val)]
test_data = [format_sample(img, lbl) for img, lbl in zip(X_test, y_test)]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"\n✅ Datasets ready: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_data)} test")

## 🧠 Model Setup (QLoRA)
Loading Qwen2-VL-2B-Instruct with 4-bit NF4 quantization and applying LoRA adapters.

In [ ]:
print("=" * 60)
print("Loading Qwen2-VL-2B-Instruct with 4-bit quantization...")
print("=" * 60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Prepare for QLoRA
model = prepare_model_for_kbit_training(model)

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.config.use_cache = False

print("\n📊 Trainable Parameters:")
model.print_trainable_parameters()

torch.cuda.empty_cache()
gc.collect()

## 🏋️ Training
Fine-tuning with SFTTrainer, gradient accumulation (effective batch=8), and mixed precision.

In [ ]:
print("=" * 60)
print("Starting training...")
print("=" * 60)

# Custom collate function for multimodal data
def collate_fn(examples):
    texts, image_inputs = [], []
    
    for example in examples:
        messages = example["messages"]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
        
        for msg in messages:
            if isinstance(msg.get("content"), list):
                for item in msg["content"]:
                    if item.get("type") == "image":
                        img = Image.open(item["image"]).convert("RGB")
                        max_dim = 1280
                        if max(img.size) > max_dim:
                            ratio = max_dim / max(img.size)
                            new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
                            img = img.resize(new_size, Image.LANCZOS)
                        image_inputs.append(img)
    
    batch = processor(
        text=texts,
        images=image_inputs if image_inputs else None,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    
    return batch

# Training args
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field=None,
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

# Train!
train_result = trainer.train()

# Save
final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(final_adapter_path)
processor.save_pretrained(final_adapter_path)

metrics = train_result.metrics
with open(os.path.join(RESULTS_DIR, "training_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n✅ Training complete!")
print(f"   Final train loss: {metrics.get('train_loss', 'N/A')}")
print(f"   Model saved to: {final_adapter_path}")

In [ ]:
# Plot training loss
if hasattr(trainer.state, "log_history") and trainer.state.log_history:
    train_losses = [x["loss"] for x in trainer.state.log_history if "loss" in x]
    eval_losses = [x["eval_loss"] for x in trainer.state.log_history if "eval_loss" in x]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#16213e")
    
    if train_losses:
        ax.plot(train_losses, color="#00D1B2", linewidth=2, label="Train Loss")
    if eval_losses:
        eval_x = np.linspace(0, len(train_losses)-1, len(eval_losses))
        ax.plot(eval_x, eval_losses, color="#FF6B6B", linewidth=2, marker="o", label="Val Loss")
    
    ax.set_title("Training & Validation Loss", fontsize=14, fontweight="bold", color="white")
    ax.set_xlabel("Step", color="white")
    ax.set_ylabel("Loss", color="white")
    ax.legend(facecolor="#16213e", edgecolor="white", labelcolor="white")
    ax.tick_params(colors="white")
    ax.grid(True, alpha=0.2)
    for spine in ax.spines.values(): spine.set_color("white")
    
    plt.savefig(os.path.join(RESULTS_DIR, "training_loss.png"), dpi=300, bbox_inches="tight", facecolor="#1a1a2e")
    plt.show()

## 📊 Evaluation
Evaluating the fine-tuned model on the held-out test set.

In [ ]:
def parse_verdict(response):
    """Parse model response to extract verdict."""
    r = response.upper()
    if "VERDICT: AUTHENTIC" in r or "VERDICT:AUTHENTIC" in r:
        return "AUTHENTIC"
    elif "VERDICT: TAMPERED" in r or "VERDICT:TAMPERED" in r:
        return "TAMPERED"
    elif "VERDICT: FORGED" in r or "VERDICT:FORGED" in r:
        return "TAMPERED"
    elif "AUTHENTIC" in r and "TAMPERED" not in r and "FORGED" not in r:
        return "AUTHENTIC"
    elif "TAMPERED" in r or "FORGED" in r:
        return "TAMPERED"
    return "UNKNOWN"

def run_inference(model, processor, image_path, prompt, device="cuda"):
    """Run inference on a single image."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": str(image_path)},
            {"type": "text", "text": prompt},
        ]},
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    img = Image.open(image_path).convert("RGB")
    max_dim = 1280
    if max(img.size) > max_dim:
        ratio = max_dim / max(img.size)
        img = img.resize((int(img.size[0]*ratio), int(img.size[1]*ratio)), Image.LANCZOS)
    
    inputs = processor(text=[text], images=[img], return_tensors="pt", padding=True).to(device)
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    
    out_trimmed = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(out_trimmed, skip_special_tokens=True)[0]

# Run evaluation
model.eval()
torch.cuda.empty_cache()
gc.collect()

y_true, y_pred, responses = [], [], []
eval_prompt = USER_PROMPTS[0]

print(f"Evaluating on {len(test_data)} test samples...")
for i, sample in enumerate(test_data):
    try:
        y_true.append(sample["label"])
        
        img_path = None
        for msg in sample["messages"]:
            if isinstance(msg.get("content"), list):
                for item in msg["content"]:
                    if item.get("type") == "image":
                        img_path = item["image"]
                        break
        
        response = run_inference(model, processor, img_path, eval_prompt)
        responses.append(response)
        
        verdict = parse_verdict(response)
        y_pred.append(0 if verdict == "AUTHENTIC" else 1)
        
        if (i+1) % 10 == 0:
            print(f"  [{i+1}/{len(test_data)}] processed...")
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"  Error on sample {i}: {e}")
        y_pred.append(0)
        responses.append(f"ERROR: {e}")

# Metrics
acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
report = classification_report(y_true, y_pred, target_names=["AUTHENTIC", "TAMPERED"], zero_division=0)

results = {
    "accuracy": float(acc), "precision_macro": float(precision),
    "recall_macro": float(recall), "f1_macro": float(f1),
    "confusion_matrix": cm.tolist(), "n_samples": len(y_true),
}

print("\n" + "=" * 60)
print("📊 EVALUATION RESULTS")
print("=" * 60)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"\n{report}")

with open(os.path.join(RESULTS_DIR, "evaluation_results.json"), "w") as f:
    json.dump(results, f, indent=2)

## 📈 Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor("#1a1a2e")

# Confusion Matrix
ax1 = axes[0]
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd",
    xticklabels=["AUTHENTIC", "TAMPERED"],
    yticklabels=["AUTHENTIC", "TAMPERED"],
    ax=ax1, annot_kws={"size": 16, "weight": "bold"})
ax1.set_title("Confusion Matrix", fontsize=14, fontweight="bold", color="white")
ax1.set_ylabel("True Label", fontsize=12, color="white")
ax1.set_xlabel("Predicted Label", fontsize=12, color="white")
ax1.tick_params(colors="white")
ax1.set_facecolor("#16213e")

# Metrics Bar Chart
ax2 = axes[1]
metric_names = ["Accuracy", "Precision", "Recall", "F1 Score"]
metric_values = [acc, precision, recall, f1]
colors = ["#00D1B2", "#7B68EE", "#FF6B6B", "#FFD93D"]
bars = ax2.bar(metric_names, metric_values, color=colors, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, metric_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
        f"{val:.2%}", ha="center", va="bottom", fontsize=12, fontweight="bold", color="white")
ax2.set_ylim(0, 1.15)
ax2.set_title("Evaluation Metrics", fontsize=14, fontweight="bold", color="white")
ax2.set_facecolor("#16213e")
ax2.tick_params(colors="white")
for spine in ["top", "right"]: ax2.spines[spine].set_visible(False)
for spine in ["bottom", "left"]: ax2.spines[spine].set_color("white")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "evaluation_plots.png"), dpi=300, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

# Sample predictions
print("\n📝 Sample Predictions:")
print("-" * 80)
for i in range(min(5, len(test_data))):
    true_lbl = "AUTHENTIC" if y_true[i] == 0 else "TAMPERED"
    pred_lbl = "AUTHENTIC" if y_pred[i] == 0 else "TAMPERED"
    correct = "✅" if y_true[i] == y_pred[i] else "❌"
    print(f"{correct} Sample {i+1}: True={true_lbl}, Pred={pred_lbl}")
    print(f"   Response: {responses[i][:120]}...")
    print()

## 💾 Save & Export
Push the trained LoRA adapter to HuggingFace Hub.

In [ ]:
# Push to HuggingFace Hub
if hf_token:
    HF_REPO = "CRGaming78/DocForge-VLM-Qwen2-2B"  # Change to your username
    
    print(f"Pushing model to {HF_REPO}...")
    model.push_to_hub(HF_REPO)
    processor.push_to_hub(HF_REPO)
    print(f"✅ Model pushed to: https://huggingface.co/{HF_REPO}")
else:
    print("⚠️ No HF token found. Set HF_TOKEN as a Kaggle secret to push to Hub.")

print("\n" + "=" * 60)
print("🎉 DocForge-VLM Pipeline Complete!")
print("=" * 60)
print(f"\nAdapter: {final_adapter_path}")
print(f"Results: {RESULTS_DIR}")